# Тюнинг токенизатора

In [ ]:
%pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 

In [ ]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

In [ ]:
dataset = load_dataset("HuggingFaceFW/fineweb-2", 'tur_Latn', split=f"train", streaming=True)


README.md:   0%|          | 0.00/844k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, GPT2TokenizerFast
from tokenizers import Tokenizer, models, trainers, pre_tokenizers


VOCAB_SIZE = 20_000 # Размер обучаемого токенизатора
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "HuggingFaceTB/SmolLM2-1.7B"
updated_tokenizer_save_path = "updated_smollm_tokenizer"

original_tokenizer = GPT2TokenizerFast.from_pretrained(model_name) # Забираю токенизатор в языковой модели
model = AutoModelForCausalLM.from_pretrained(model_name) # сама языковая модель с CasualLM-головой
model.to(device)

print(f"Оригинальный размер словаря: {len(original_tokenizer)}")

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Оригинальный размер словаря: 49152


In [ ]:
import re

def clean_sentence(sentence):
    return re.sub(
        r'[^a-zA-Z0-9ğüşöçıĞÜŞÖÇİ\s\n\r`",.;:!?\'"()/@&#%+=*/-]',
        '',
        sentence
    )

In [ ]:
from tqdm import tqdm
def batch_iterator(dataset):
    for i in tqdm(dataset):
        yield clean_sentence(i["text"])

In [ ]:
original_tokenizer

GPT2TokenizerFast(name_or_path='HuggingFaceTB/SmolLM2-1.7B', vocab_size=49152, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'additional_special_tokens': ['<|endoftext|>', '<|im_start|>', '<|im_end|>', '<repo_name>', '<reponame>', '<file_sep>', '<filename>', '<gh_stars>', '<issue_start>', '<issue_comment>', '<issue_closed>', '<jupyter_start>', '<jupyter_text>', '<jupyter_code>', '<jupyter_output>', '<jupyter_script>', '<empty_output>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_n

In [ ]:
# Создаем новый токенизатор с базовой моделью BPE ( в SmolLM используется GPT2Fast ~ BPE )
new_tokenizer = Tokenizer(models.BPE()) # Инициализирую новый токенизатор
new_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace() # Претокенизация через пробелы
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[], show_progress=True) # Создаю экземпляр для обучения токенизатора
new_tokenizer.train_from_iterator(batch_iterator(dataset), trainer=trainer) # Обучение токенизатора ( Занимает много времени и зависит от CPU )


598309it [08:34, 1353.09it/s]

In [ ]:
new_vocab = new_tokenizer.get_vocab() # Получаю вокабуляр из токенизатора
new_tokens = set(new_vocab.keys()) # Получаю уникальные токены из токенизатора
original_vocab = original_tokenizer.get_vocab() # Получаю вокабуляр из оригинального токенизатора
unique_new_tokens = [token for token in new_tokens if token not in original_vocab] # Создаю список из новых токенов, которых не было в оригинальном токенизаторе
print(f"Найдено {len(unique_new_tokens)} новых токенов для добавления.")



Найдено 17760 новых токенов для добавления.


In [ ]:
from copy import deepcopy
new_tokenizer = deepcopy(original_tokenizer)
added_count = new_tokenizer.add_tokens(unique_new_tokens) # Добавляю токены в изначальный токенизатор
print(f"Добавлено {added_count} новых токенов в исходный токенизатор.")



Добавлено 17760 новых токенов в исходный токенизатор.


In [ ]:
# Изменение размеров эмбеддингов модели
model.resize_token_embeddings(len(new_tokenizer)) # Изменяю размерность эмбедингов под токены
embedding_layer = model.get_input_embeddings() # Получаю новые эмбединги



The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
embedding_layer.weight[49999]

tensor([ 0.0054,  0.0084,  0.0141,  ..., -0.0092,  0.0002, -0.0088],
       device='cuda:0', grad_fn=<SelectBackward0>)

In [ ]:
def smart_initialize(token, tokenizer):
    """
    Разбивает новый токен на под-токены с использованием оригинального токенизатора.
    Если под-токены найдены, возвращает их среднее эмбеддингов, иначе — случайный вектор.
    """
    # Используем метод tokenize для разбиения нового токена
    sub_tokens = tokenizer.tokenize(token)
    # Для каждого под-токена получаем его ID, если он присутствует в оригинальном словаре
    sub_ids = [tokenizer.convert_tokens_to_ids(st) for st in sub_tokens if st in original_vocab]
    if sub_ids:
        sub_embs = embedding_layer.weight.data[sub_ids]
        return sub_embs.mean(dim=0)
    else:
        # Если не найдено подходящих под-токенов, возвращаем случайный вектор
        return torch.randn(embedding_layer.weight.size(1), device=device)

# Инициализируем эмбеддинги для каждого нового токена
for token in tqdm(unique_new_tokens):
    token_id = new_tokenizer.convert_tokens_to_ids(token)
    new_emb = smart_initialize(token, original_tokenizer)
    embedding_layer.weight.data[token_id] = new_emb


# Сохранение обновленного токенизатора
new_tokenizer.save_pretrained(updated_tokenizer_save_path)
print(f"Обновленный токенизатор сохранен в {updated_tokenizer_save_path}")


100%|██████████| 17760/17760 [00:01<00:00, 11656.23it/s]


Обновленный токенизатор сохранен в updated_smollm_tokenizer


In [ ]:
import re
from transformers import PreTrainedTokenizerFast, GPT2TokenizerFast
from tokenizers.models import BPE
from tokenizers import Tokenizer


updated_tokenizer = "updated_smollm_tokenizer"

tokenizer = GPT2TokenizerFast.from_pretrained(updated_tokenizer)


In [ ]:
vocab = tokenizer.get_vocab()

In [ ]:
allowed_pattern = re.compile(r'^[a-zA-Z0-9!"#$%&\'()*+,-./:;<=>?@[\\\]^_`{|}~\sçÇğĞöÖşŞüÜıİğüşöçİĞÜŞÖÇ]+$')

In [ ]:
def is_allowed(token):
    if token.startswith("<") and token.endswith(">"):
        return True
    if token.startswith("Ġ"):
        return True
    return bool(allowed_pattern.match(token))

In [ ]:
clean_tokens = [token for token in vocab.keys() if is_allowed(token)]
not_valid_tokens = [token for token in vocab.keys() if not is_allowed(token)]
print(f"Оставляем {len(clean_tokens)} токенов из {len(vocab)}")

Оставляем 66018 токенов из 66912


In [ ]:
new_vocab = {token: idx for idx, token in enumerate(clean_tokens)}

In [ ]:
special_tokens = list(tokenizer.special_tokens_map.values())
for sp_token in special_tokens[3]:
    if sp_token not in new_vocab:
        new_vocab[sp_token] = len(new_vocab)

In [ ]:
tokenizer._tokenizer.model.vocab = new_vocab


AttributeError: 'tokenizers.models.BPE' object has no attribute '_vocab'

In [ ]:
!pip install -q transformers datasets peft bitsandbytes flash-attn trl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flagembedding 1.3.3 requires datasets==2.19.0, but you have datasets 3.4.0 which is incompatible.
flagembedding 1.3.3 requires transformers==4.44.2, but you have transformers 4.49.0 which is incompatible.
datatrove 0.4.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
from tqdm import tqdm

In [ ]:
SEED = 42

In [ ]:
import pandas as pd

eng_df = pd.read_feather('fineweb_english.feather')
tur_df = pd.read_feather('fineweb_turkish.feather')

merged_df = pd.concat([eng_df, tur_df], ignore_index=True)

In [ ]:
from datasets import load_dataset, Dataset


dataset = Dataset.from_pandas(merged_df)

dataset = dataset.shuffle(seed=SEED)
valid_data = dataset.take(4000)
train_data = dataset.skip(4000)

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

model_name = "HuggingFaceTB/SmolLM2-1.7B"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path='updated_smollm_tokenizer')

model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-Turkish"


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
def chars_token_ratio(dataset, tokenizer, data_column, nb_examples=200):
    """
    Оценка качества токенизации. Хорошим соотношением считается 2 - 3.5 символа на токен
    """

    total_characters, total_tokens = 0, 0
    for _, example in tqdm(zip(range(nb_examples), iter(dataset)), total=nb_examples):
        total_characters += len(example[data_column])
        total_tokens += len(tokenizer(example[data_column]).tokens())

    return total_characters / total_tokens


chars_per_token = chars_token_ratio(valid_data, tokenizer, 'text', 5000)
print(f"The character to token ratio of the dataset is: {chars_per_token:.2f}")

 80%|████████  | 4000/5000 [00:09<00:02, 442.68it/s]

The character to token ratio of the dataset is: 3.10


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # Авторегрессия (GPT-2, GPT-3)
    inference_mode=False,
    r=16,  # Размер низкорангового представления (можно увеличить для более качественного обучения)
    lora_alpha=32,  # Коэффициент масштабирования
    lora_dropout=0.05  # Dropout
)


In [ ]:
model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(66912, 2048)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb)

In [ ]:
model = get_peft_model(model, lora_config)


In [ ]:
model.print_trainable_parameters()


trainable params: 3,145,728 || all params: 1,750,894,592 || trainable%: 0.1797


In [ ]:
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=5, # number of epochs
    per_device_train_batch_size=4, # batch size for training per GPU or core CPU
    per_device_eval_batch_size=4, # batch size for evaluating per GPU or core CPU
    gradient_accumulation_steps=4, # number of updates steps to accumulate the gradients for, before performing a backward/update pass
    save_steps=50, # number of updates steps before two checkpoint saves
    logging_steps=50, # number of update steps between two logs
    learning_rate=5e-5, # 0.00005 initial learning rate for [`AdamW`] optimizer
    lr_scheduler_type="constant", # the scheduler type to use
    warmup_ratio=0.03, # ratio of total training steps used for a linear warmup from 0 to `learning_rate`
    weight_decay=0.001, # the weight decay to apply to all layers except all bias and LayerNorm weights in [`AdamW`] optimizer
    fp16=True, # whether to use fp16 16-bit (mixed) precision training instead of 32-bit training
    bf16=False, # whether to use bfp16 16-bit (mixed) precision training instead of 32-bit training
    max_grad_norm=0.3, # maximum gradient norm (for gradient clipping)
    max_steps=-1, # if set to a positive number, the total number of training steps to perform. Overrides `num_train_epochs`    group_by_length=True, # Whether or not to group together samples of roughly the same length in the training dataset
    gradient_checkpointing=True,
    optim='adamw_torch',
    report_to="none",
)



In [ ]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_data,
    tokenizer=tokenizer,
    eval_dataset=valid_data,
    callbacks=[]
)



/tmp/ipykernel_200348/2146594751.py:2: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Converting train dataset to ChatML:   0%|          | 0/96000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/96000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/96000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10994 > 8192). Running this sequence through the model will result in indexing errors


...

Converting eval dataset to ChatML:   0%|          | 0/4000 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
model.enable_input_require_grads()


In [ ]:
trainer.train()

Step,Training Loss
50,7.091700
100,6.204500
150,5.845900
200,5.747400
250,5.721500
300,5.664100
350,5.660900
400,5.626500
450,5.613600
500,5.528500


/opt/conda/lib/python3.11/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /HuggingFaceTB/SmolLM2-1.7B/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7fa059d7c750>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: ee80ca47-6457-4db0-8407-9070e94769e4)') - silently ignoring the lookup for the file config.json in HuggingFaceTB/SmolLM2-1.7B.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/peft/utils/save_and_load.py:246: UserWarning: Could not find a config file in HuggingFaceTB/SmolLM2-1.7B - will assume that the vocabulary was not modified.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='hugg

In [ ]:
!tar -cvf results.tar results/checkpoint-30000

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


results/checkpoint-30000/
results/checkpoint-30000/README.md
results/checkpoint-30000/adapter_model.safetensors
results/checkpoint-30000/adapter_config.json
results/checkpoint-30000/tokenizer_config.json
results/checkpoint-30000/special_tokens_map.json
results/checkpoint-30000/added_tokens.json
results/checkpoint-30000/vocab.json
results/checkpoint-30000/merges.txt
results/checkpoint-30000/tokenizer.json
results/checkpoint-30000/training_args.bin
results/checkpoint-30000/optimizer.pt
results/checkpoint-30000/scheduler.pt
results/checkpoint-30000/scaler.pt
results/checkpoint-30000/rng_state.pth
results/checkpoint-30000/trainer_state.json
results/checkpoint-30000/.ipynb_checkpoints/
results/checkpoint-30000/.ipynb_checkpoints/README-checkpoint.md
results/checkpoint-30000/.ipynb_checkpoints/trainer_state-checkpoint.json
results/checkpoint-30000/.ipynb_checkpoints/tokenizer-checkpoint.json
